# LOOKLAB Native Sharp — 1024 × 1024
ปรับจากไฟล์เดิม HairFastGAN B24 + Diffusion ไม่ใช้ Real-ESRGAN / Swin2SR

- คงโมเดล, prompt, strength .15, steps 30, guidance 2.5 และขนาด Diffusion 1024 เดิม
- เพิ่มความคมแบบ luminance เฉพาะผมและหน้า หน้าใช้ความแรงต่ำกว่าผม
- จำกัดการเปลี่ยนพิกเซลและลดความแรงใกล้ขอบ; ไม่เพิ่มรอบ Diffusion หรือ segmentation
- ไม่ใช่การสร้างรายละเอียดจริงที่สูญหายกลับมา และไม่รับประกันว่าทุกภาพจะดีขึ้น
- ส่วน 12 แสดง B24 / Raw Diffusion / ผลเดิม / ผลเพิ่มความคม และ crop ตำแหน่งเดียวกัน
- เซลล์ดาวน์โหลดบันทึก PNG แยก 4 รูป ไม่บีบเป็น ZIP
- ท้ายไฟล์ปรับความแรงบนผลเดิมได้ โดยไม่รันโมเดลใหม่
- Runtime ใหม่: รันส่วน 1–7 แล้วส่วน 12 เพื่อทดสอบ; ส่วน API เลือกเปิดเมื่อต้องการใช้เว็บ
- Runtime เดิมที่โหลดโมเดลครบแล้ว: รันส่วน 6 และ 7 ใหม่ แล้วส่วน 12; หากใช้เว็บให้รันส่วน 9–11 ใหม่
- ตั้ง NATIVE_SHARP_ENABLED=False ในส่วน 6 เพื่อคืนผลเดิม
- ตรวจ syntax และฟังก์ชันเพิ่มความคมด้วยภาพจำลองแล้ว ยังไม่ได้ทดสอบโมเดลบน GPU หรือยืนยันคุณภาพกับภาพจริง


# B24 full image + diffusion เฉพาะพื้นที่แก้
มาสก์กำหนดจุดปรับ ไม่ใช้ตัดทรงผม


## 1. ติดตั้ง

In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub requests
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


## 2. เตรียม repository และ pretrained models

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download

drive.mount('/content/drive')
LOCAL_REPO = '/content/HairFastGAN'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
RESULTS_ROOT = Path('/content/drive/MyDrive/HairFastGAN_B24_HairRefine_FaceKeep')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
        shutil.copytree(local_models, DRIVE_CACHE_DIR, dirs_exist_ok=True)
# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


## 3. Backend ที่ไม่ต้อง compile CUDA extension (แบบเดียวกับ notebook ฝึก)

In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))

# Small on-device smoke check before loading model weights.
import torch
_check = {'torch':torch, 'F':torch.nn.functional}
exec(FUSED_NATIVE + '\n' + UP_NATIVE_IMPL + '\n' + UP_NATIVE, _check)
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_x = torch.randn(1, 2, 5, 5, device=_device, requires_grad=True)
_b = torch.randn(2, device=_device, requires_grad=True)
_y = _check['fused_leaky_relu'](_x, _b, .1)
torch.testing.assert_close(_y, torch.nn.functional.leaky_relu(_x + _b.view(1,2,1,1), .1) * (2 ** .5))
_z = _check['upfirdn2d'](_x, torch.ones(1,1,device=_device))
torch.testing.assert_close(_z, _x)
(_y.sum() + _z.sum()).backward()
assert torch.isfinite(_x.grad).all() and torch.isfinite(_b.grad).all()
print('Native operator smoke check passed on', _device)
del _check, _x, _b, _y, _z


## 4. โหลด B epoch 24

In [ ]:
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')

import hashlib, inspect, textwrap, types
CHECKPOINT = Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart/9be83b9c2e/trial_B/best.pt')
EXPECTED_SHA256 = '6195edd247917e7c34887945413e08b731faae736f5239dd2b28d9636e65f98b'
def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
if not CHECKPOINT.is_file():
    raise FileNotFoundError('ไม่พบ B best.pt ใน Drive: '+str(CHECKPOINT))
if sha_file(CHECKPOINT)!=EXPECTED_SHA256:raise RuntimeError('ไฟล์ไม่ใช่ B epoch 24 ที่ตรวจไว้')
ck=torch.load(CHECKPOINT,map_location='cpu',weights_only=True)
assert ck['epoch']==24 and ck['config']['trial']=='B'
b=hair_fast.blend
assert sha_file(b.opts.blending_checkpoint)==ck['start_blending_sha256']
enc=b.blending_encoder
expected={k for k,_ in enc.named_parameters() if k.startswith('modulation_module_list.')}
assert set(ck['blending_trainable_state_dict'])==expected
state=enc.state_dict();state.update(ck['blending_trainable_state_dict'])
enc.load_state_dict(state,strict=True)
b.post_process.load_state_dict(ck['post_process_state_dict'],strict=True)
enc.eval();b.post_process.eval()
del state,ck
# Match the training cache: mixing .95; shape rotation OFF; color rotation ON.
for obj in (hair_fast.args,hair_fast.embed.opts,hair_fast.align.opts,b.opts):
    obj.mixing=.75;obj.save_all=False
method=hair_fast.align.shape_module
raw=textwrap.dedent(inspect.getsource(method))
raw='\n'.join(line for line in raw.splitlines() if not line.startswith('@'))
anchor='    # Rotate stage\n    if img1_in is not img2_in:'
replacement=('    # Shape OFF, color ON, same policy as training\n'
             '    if im_name2 == "shape":\n'
             '        I_rot = None\n        rot_mask = inp_mask2\n'
             '    elif img1_in is not img2_in:')
assert raw.count(anchor)==1,'Alignment source ไม่ตรงรุ่น'
ns = dict(inspect.unwrap(method.__func__).__globals__)
exec(compile(raw.replace(anchor,replacement),'<shape_rotation_policy>','exec'),ns)
hair_fast.align.shape_module=types.MethodType(torch.inference_mode()(ns['shape_module']),hair_fast.align)
print('โหลด B epoch 24 ครบ: Blending + Post-process | mix .95 | shape rotate OFF / color ON')


## 5. โหลด segmentation และ diffusion

In [ ]:
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageDraw
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from diffusers import StableDiffusionInpaintPipeline, UniPCMultistepScheduler

DEVICE = 'cuda'
SEG_MODEL_ID = 'mattmdjaga/segformer_b2_clothes'
DIFFUSION_MODEL_ID = 'stablediffusionapi/realistic-vision-v51'
SEG_REVISION = 'main'           # เปลี่ยนเป็น commit hash เพื่อทำซ้ำข้าม session
DIFFUSION_REVISION = 'main'
DIFFUSION_CPU_OFFLOAD = True
seg_key=(SEG_MODEL_ID,SEG_REVISION,DEVICE)
if globals().get('_SEG_LOADED_KEY') != seg_key or 'seg_model' not in globals():
    seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID, revision=SEG_REVISION)
    seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID, revision=SEG_REVISION).to(DEVICE).eval()
    _SEG_LOADED_KEY=seg_key
else:
    print('ใช้ segmentation ที่อยู่ใน RAM แล้ว')
label_ids = {str(name).lower(): int(i) for i, name in seg_model.config.id2label.items()}
HAIR_ID, FACE_ID = label_ids['hair'], label_ids['face']
assert (HAIR_ID, FACE_ID) == (2, 11), 'ตรวจ mapping ของโมเดลก่อนทดลอง'
diffusion_key=(DIFFUSION_MODEL_ID,DIFFUSION_REVISION,DIFFUSION_CPU_OFFLOAD,DEVICE)
if globals().get('_DIFFUSION_LOADED_KEY') != diffusion_key or 'inpaint_pipe' not in globals():
    # # ไม่ปิด safety checker ที่โมเดลมีมา
    # inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    #     DIFFUSION_MODEL_ID, revision=DIFFUSION_REVISION, torch_dtype=torch.float16)

    # ปิด Safety Checker
    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        DIFFUSION_MODEL_ID,
        revision=DIFFUSION_REVISION,
        torch_dtype=torch.float16,
        safety_checker=None,
    )

    inpaint_pipe.scheduler = UniPCMultistepScheduler.from_config(inpaint_pipe.scheduler.config)
    if DIFFUSION_CPU_OFFLOAD:
        inpaint_pipe.enable_model_cpu_offload()
    else:
        inpaint_pipe.to(DEVICE)
    # รองรับทั้ง pipeline API เดิม และ VAE API; ถ้าไม่มีให้ข้าม optimization นี้
    vae = getattr(inpaint_pipe, 'vae', None)
    if callable(getattr(vae, 'enable_slicing', None)):
        vae.enable_slicing()
    elif callable(getattr(inpaint_pipe, 'enable_vae_slicing', None)):
        inpaint_pipe.enable_vae_slicing()
    else:
        print('VAE slicing ไม่รองรับในรุ่นนี้: ใช้งานต่อโดยไม่เปิด slicing')
    _DIFFUSION_LOADED_KEY=diffusion_key
else:
    print('ใช้ diffusion pipeline ที่โหลดไว้แล้ว')
print('Labels:', HAIR_ID, FACE_ID, '| UNet channels:', inpaint_pipe.unet.config.in_channels)
# บันทึกรุ่นที่ใช้จริง ไม่เก็บ secrets
ENVIRONMENT = {
    'repo_commit': REPO_COMMIT, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda,
    'seg_model': SEG_MODEL_ID, 'seg_revision_requested': SEG_REVISION,
    'seg_revision_resolved': getattr(seg_model.config, '_commit_hash', None),
    'diffusion_model': DIFFUSION_MODEL_ID, 'diffusion_revision_requested': DIFFUSION_REVISION,
    'unet_channels': inpaint_pipe.unet.config.in_channels,
    'scheduler': type(inpaint_pipe.scheduler).__name__,
    'cpu_offload': DIFFUSION_CPU_OFFLOAD,
}
(RESULTS_ROOT / 'requirements.freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## 6. Diffusion เดิม + ปรับพื้นหลัง + เพิ่มความคมผมและหน้าแบบเบา
ปรับ NATIVE_HAIR_AMOUNT / NATIVE_FACE_AMOUNT ที่ท้ายเซลล์นี้


In [ ]:
import io, threading, uuid, zipfile
from scipy.ndimage import binary_fill_holes, distance_transform_edt
GPU_LOCK=threading.Lock()
DEFAULT_SEED=3407
INPAINT_STRENGTH=.15
INPAINT_STEPS=30
INPAINT_GUIDANCE=2.5
JOINT_FACE_BLEND=0.35
BACKGROUND_BLEND=0.0  # พื้นหลังซ่อมแยก ไม่ผ่าน hair/face diffusion
BACKGROUND_REPAIR_MODE="auto"  # auto: สีเรียบเมื่อมั่นใจ มิฉะนั้น inpaint
BACKGROUND_REPAIR_STRENGTH=0.65
BACKGROUND_REPAIR_STEPS=40
BACKGROUND_REPAIR_FEATHER_PX=24.0
BACKGROUND_REPAIR_EXTRA_MASK=""  # optional: PNG mask ขนาดเท่า aligned image; white=repair
DIFFUSION_SIDE=1024
JOINT_TRANSITION_PX=24
JOINT_MARGIN_PX=12
JOINT_EYE_MARGIN_PX=3
JOINT_EYE_FEATHER_PX=6
_EYE_LANDMARK_MODEL=globals().get('_EYE_LANDMARK_MODEL')
_EYE_INIT_LOCK=globals().get('_EYE_INIT_LOCK',threading.Lock())
_DLIB_DETECTOR=globals().get('_DLIB_DETECTOR')
# Limited seam edits share the existing diffusion pass.
SEAM_WIDTH_PX=4.0
SEAM_BLEND=0.35
HAIR_DIFFUSION_BLEND=1.0
PROMPT = ('Natural portrait photograph, same person, haircut and hair color. '
          'Real hair with varied strand thickness, irregular fine strands, subtle flyaways, '
          'soft directional highlights and depth between locks. Natural skin texture, '
          'same eyes and gaze. Consistent lighting, natural hair edges against the existing background.')
NEGATIVE = ('wig, plastic hair, painted streaks, uniform parallel strands, flat clumps, '
            'cutout edges, bright halo, oversharpening, blur, waxy skin, '
            'changed face, changed haircut, changed hair color, new background objects')
for text in (PROMPT,NEGATIVE):
    assert len(inpaint_pipe.tokenizer(text,truncation=False)['input_ids']) <= inpaint_pipe.tokenizer.model_max_length,'Prompt ยาวเกิน tokenizer'
def segment_labels(image):
    inputs = seg_processor(images=image.convert('RGB'), return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        logits = seg_model(**inputs).logits
        logits = torch.nn.functional.interpolate(logits, size=image.size[::-1], mode='bilinear', align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)

def composite_protected(base, edited, alpha):
    if base.size != edited.size or np.asarray(alpha).shape != (base.height, base.width):
        raise ValueError('ภาพและ alpha ต้องมีขนาดตรงกัน')
    a = np.clip(np.asarray(alpha, dtype=np.float32), 0, 1)[..., None]
    original = np.array(base.convert('RGB'))
    changed = np.asarray(edited.convert('RGB'), dtype=np.float32)
    output = np.rint(original * (1 - a) + changed * a).clip(0, 255).astype(np.uint8)
    output[a[..., 0] == 0] = original[a[..., 0] == 0]
    return Image.fromarray(output)

def padded(image, target=512, mask=False):
    scale = target / max(image.size)
    size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
    resized = image.resize(size, Image.Resampling.NEAREST if mask else Image.Resampling.LANCZOS)
    canvas = Image.new(image.mode, (target, target), 0)
    x, y = (target-size[0])//2, (target-size[1])//2
    canvas.paste(resized, (x, y))
    return canvas, (x, y, x+size[0], y+size[1])

def tensor_to_pil(value, value_range='zero_one'):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if torch.is_tensor(value):
        t = value.detach().cpu()
        if t.ndim == 4 and t.shape[0] == 1:
            t = t[0]
        if t.ndim != 3 or t.shape[0] != 3:
            raise ValueError(f'Unexpected tensor shape: {tuple(t.shape)}')
        a = t.permute(1, 2, 0).numpy()
    else:
        a = np.asarray(value)
    if a.dtype == np.uint8:
        return Image.fromarray(a).convert('RGB')
    if value_range == 'minus_one_one':
        a = (a + 1) / 2
    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:
        raise ValueError('Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ')
    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert('RGB')

def resolve_inpaint_settings(strength=None, steps=None, guidance=None):
    import math
    strength = float(INPAINT_STRENGTH if strength is None else strength)
    steps = INPAINT_STEPS if steps is None else steps
    guidance = float(INPAINT_GUIDANCE if guidance is None else guidance)
    if isinstance(steps, bool) or not isinstance(steps, (int, np.integer)) or not 1 <= steps <= 100:
        raise ValueError('steps ต้องเป็นจำนวนเต็ม 1–100')
    steps = int(steps)
    if not math.isfinite(strength) or not 0 < strength <= 1:
        raise ValueError('strength ต้องมากกว่า 0 และไม่เกิน 1')
    if not math.isfinite(guidance) or not 1 <= guidance <= 20:
        raise ValueError('guidance ต้องอยู่ในช่วง 1–20')
    if int(steps * strength) < 1:
        raise ValueError('steps × strength ต้องไม่น้อยกว่า 1')
    return {'strength':strength, 'steps':steps, 'guidance':guidance}

def resolve_joint_settings(face_blend=JOINT_FACE_BLEND):
    import math
    face_blend = float(face_blend)
    if not math.isfinite(face_blend) or not 0 <= face_blend <= 1:
        raise ValueError('face_blend ต้องอยู่ในช่วง 0–1')
    return {'face_blend':face_blend}

def prepare_eye_models():
    """One instance per runtime, including repeated execution of this cell."""
    global _EYE_LANDMARK_MODEL, _DLIB_DETECTOR
    import face_alignment, inspect, dlib, hashlib, shutil
    from pathlib import Path
    with _EYE_INIT_LOCK:
        if _EYE_LANDMARK_MODEL is not None:
            if _DLIB_DETECTOR is None:_DLIB_DETECTOR=dlib.get_frontal_face_detector()
            print('ใช้โมเดลตรวจหน้า–ตาที่อยู่ใน RAM แล้ว')
            return _EYE_LANDMARK_MODEL
        # Keep Torch's normal LOCAL cache for speed; mirror checkpoints to Drive.
        local=Path(torch.hub.get_dir())/'checkpoints'
        persistent=Path('/content/drive/MyDrive/HairFastGAN_cache/face_alignment')
        local.mkdir(parents=True,exist_ok=True)
        drive_ready=Path('/content/drive/MyDrive').is_dir()
        manifest_path=persistent/'manifest.json'
        if drive_ready:
            persistent.mkdir(parents=True,exist_ok=True)
            manifest=json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
            for name,entry in manifest.items():
                if Path(name).name!=name:continue
                src=persistent/name;dst=local/name
                if dst.exists() or not src.is_file():continue
                if src.stat().st_size!=entry['size']:continue
                digest=hashlib.sha256(src.read_bytes()).hexdigest()
                if digest!=entry['sha256']:continue
                tmp=dst.with_name(dst.name+'.copying');shutil.copyfile(src,tmp);tmp.replace(dst)
                print('นำ checkpoint จาก Drive มาใช้:',name)
        before={f.name for f in local.iterdir() if f.is_file()}
        kwargs=dict(device='cpu',flip_input=False)
        if 'compile' in inspect.signature(face_alignment.FaceAlignment).parameters:
            kwargs['compile']=False
        model=face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D,**kwargs)
        detector=dlib.get_frontal_face_detector()
        # Cache both known FAN/S3FD files and any new files downloaded by this version.
        if drive_ready:
            manifest=json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
            for src in local.iterdir():
                if not src.is_file():continue
                if src.name in before and not any(k in src.name.lower() for k in ('s3fd','2dfan','3dfan')):continue
                if not src.name.endswith(('.pth','.pth.tar','.pt')):continue
                entry=dict(size=src.stat().st_size,sha256=hashlib.sha256(src.read_bytes()).hexdigest())
                if manifest.get(src.name)!=entry or not (persistent/src.name).is_file():
                    tmp=persistent/(src.name+'.copying');shutil.copyfile(src,tmp);tmp.replace(persistent/src.name)
                manifest[src.name]=entry
            tmp=manifest_path.with_suffix('.tmp');tmp.write_text(json.dumps(manifest,indent=2));tmp.replace(manifest_path)
        _EYE_LANDMARK_MODEL=model;_DLIB_DETECTOR=detector
        print('โมเดลตรวจหน้า–ตาพร้อม ใช้ซ้ำทุกภาพใน runtime นี้')
        return model

def detect_eye_landmarks(base):
    """Inference only after setup; no repeated model construction."""
    if _EYE_LANDMARK_MODEL is None:
        raise RuntimeError('รันส่วน 6 ให้จบก่อน เพื่อเตรียมโมเดลตรวจหน้า–ตา')
    def validate(predictions, size):
        if predictions is None or len(predictions) != 1: return None
        p = np.asarray(predictions[0], dtype=np.float32).copy()
        if p.shape != (68, 2) or not np.isfinite(p).all(): return None
        p[:, 0] *= base.width / size[0]
        p[:, 1] *= base.height / size[1]
        eyes = p[36:48]
        if (eyes < 0).any() or (eyes[:,0] >= base.width).any() or (eyes[:,1] >= base.height).any(): return None
        if any(np.ptp(p[a:b,0]) < 2 for a,b in ((36,42),(42,48))): return None
        return p
    detector = _DLIB_DETECTOR
    with torch.inference_mode():
        predictions = _EYE_LANDMARK_MODEL.get_landmarks(np.asarray(base.convert('RGB')))
        p = validate(predictions, base.size)
        if p is not None: return p
        # Detection-only resizing/contrast. Never alter the image sent to diffusion.
        for side in (512, 768):
            scale = side / max(base.size)
            size = (max(1, round(base.width*scale)), max(1, round(base.height*scale)))
            im = base.convert('RGB').resize(size, Image.Resampling.LANCZOS)
            rgb = np.ascontiguousarray(np.asarray(im))
            for gray in (im.convert('L'), ImageOps.autocontrast(im.convert('L'))):
                rects = detector(np.ascontiguousarray(np.asarray(gray)), 1)
                if len(rects) != 1: continue
                r = rects[0]
                boxes = [[max(0,r.left()),max(0,r.top()),min(size[0]-1,r.right()),min(size[1]-1,r.bottom())]]
                predictions = _EYE_LANDMARK_MODEL.get_landmarks(rgb, detected_faces=boxes)
                p = validate(predictions, size)
                if p is not None:
                    print('Eye protection: ใช้ตัวตรวจสำรองสำเร็จ')
                    return p
    raise ValueError('ยังตรวจตำแหน่งตาที่เชื่อถือได้ไม่ได้ หยุดก่อน diffusion; ส่งภาพ hairfast_B24.png จากโฟลเดอร์ผลเพื่อตรวจต่อ')

def build_eye_protection(size, landmarks):
    """eye_keep=1 คืน A เต็มที่; 0 ใช้ Joint ตามเดิม; ขอบมีช่วงเปลี่ยนที่จำกัด."""
    from scipy.ndimage import distance_transform_edt
    w, h = size
    canvas = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(canvas)
    for eye in (landmarks[36:42], landmarks[42:48]):
        if np.ptp(eye[:, 0]) < 2 or (eye < 0).any() or (eye[:, 0] >= w).any() or (eye[:, 1] >= h).any():
            raise ValueError('Eye protection: ตำแหน่งตาอยู่นอกภาพหรือเล็กเกินไป')
        draw.polygon([tuple(map(float, p)) for p in eye], fill=255)
    scale = min(w, h) / 512.0
    margin = max(0, round(JOINT_EYE_MARGIN_PX * scale))
    if margin:
        canvas = canvas.filter(ImageFilter.MaxFilter(2 * margin + 1))
    core = np.asarray(canvas) > 0
    feather = max(1.0, JOINT_EYE_FEATHER_PX * scale)
    x = np.clip(distance_transform_edt(~core) / feather, 0, 1)
    keep = (1 - x*x*(3-2*x)).astype(np.float32)
    keep[core] = 1.0
    return {'eye_core': core, 'eye_keep': keep}


# ความกว้างรอยต่อ หน่วยพิกเซลที่ภาพ 1024; ปรับตามขนาดภาพอัตโนมัติ
RESTORE_FEATHER_PX = 16.0
HAIR_FACE_FEATHER_PX = 6.0
HAIR_BACKGROUND_FEATHER_PX = 4.0
LABEL_TRANSITION_PX = 8.0

def smootherstep(t):
    t=np.clip(t,0,1).astype(np.float32)
    return t*t*t*(t*(t*6-15)+10)

def outside_distance(mask):
    mask=np.asarray(mask,dtype=bool)
    if not mask.any():return np.full(mask.shape,np.inf,dtype=np.float32)
    return distance_transform_edt(~mask).astype(np.float32)

def outward_feather(mask,width):
    # น้ำหนัก 1 ใน mask แล้วค่อยลดลงถึง 0 ภายนอก ไม่มี Gaussian tail
    return 1-smootherstep(outside_distance(mask)/np.maximum(width,1.0))

def background_confidence(image):
    """Two views of the existing parser; no additional model downloads."""
    bg_id=label_ids['background']
    views=[];base_labels=None
    for flip in (False,True):
        view=ImageOps.mirror(image) if flip else image
        inputs=seg_processor(images=view.convert('RGB'),return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            logits=seg_model(**inputs).logits
            logits=torch.nn.functional.interpolate(logits,size=image.size[::-1],mode='bilinear',align_corners=False)
            if not flip:base_labels=logits.argmax(1)[0].cpu().numpy().astype(np.uint8)
            probs=logits.softmax(1)[0,[bg_id,HAIR_ID]].cpu().numpy()
        if flip:probs=probs[:,:,::-1].copy()
        views.append(probs)
    # Confidence is a heuristic, not a calibrated probability of correctness.
    bg=np.minimum(views[0][0],views[1][0])
    hair=np.maximum(views[0][1],views[1][1])
    return bg,hair,base_labels

def preservation_and_repair_masks(base,original,lo,lb,background_score,hair_score):
    from scipy.ndimage import binary_dilation
    scale=min(base.size)/1024.;bg_id=label_ids['background']
    old=lo==HAIR_ID
    # Separate liberal keep mask from narrow diffusion hair mask.
    hair_keep=(lb==HAIR_ID)|(hair_score>=.08)
    # New pixels on source background may be missed new hair: preserve them.
    delta=np.linalg.norm(np.asarray(base,dtype=np.float32)-np.asarray(original,dtype=np.float32),axis=2)
    uncertain_new=(lo==bg_id)&(delta>18)
    keep=hair_keep|uncertain_new|(lb!=bg_id)|(background_score<.98)
    guard=binary_dilation(keep,iterations=max(1,round(6*scale)))
    old_zone=binary_dilation(old,iterations=max(1,round(20*scale)))
    candidate=old_zone&(lb==bg_id)&(background_score>=.98)&~guard
    return guard,candidate

def prepare_face_base(base,original,need_landmarks=True):
    if base.size!=original.size:raise ValueError('aligned source กับ B24 ขนาดไม่ตรงกัน')
    lo=segment_labels(original)
    bg_score,hair_score,lb=background_confidence(base)
    old,new=lo==HAIR_ID,lb==HAIR_ID
    if not old.any() or not new.any() or not (lo==FACE_ID).any():
        raise ValueError('ตรวจผมหรือหน้าไม่ได้ โปรดตรวจ segmentation')
    # Full B24 base: no source face is pasted, so no source alignment detector needed.
    if need_landmarks:
        if _EYE_LANDMARK_MODEL is None:prepare_eye_models()
        lm1=detect_eye_landmarks(base)
    else:
        lm1=None
    scale=min(base.size)/1024.
    # Segmentation selects edits only, NEVER cuts the B24 silhouette.
    # Preserve every B24 pixel before diffusion, including missed hair strands.
    patch=np.ones((base.height,base.width),dtype=np.float32)
    restored=base.copy()
    keep_guard,repair_candidate=preservation_and_repair_masks(base,original,lo,lb,bg_score,hair_score)
    eyes=(build_eye_protection(base.size,lm1) if lm1 is not None else
          {'eye_core':np.zeros(lb.shape,bool),'eye_keep':np.zeros(lb.shape,np.float32)})
    # ลดการป้องกันอย่างต่อเนื่องเมื่อมีผมบัง ไม่ใช้ boolean ตัดขอบตา
    eye_keep=(eyes['eye_keep']*(1-patch)).astype(np.float32)
    visible_eyes=eyes['eye_core'] & (patch==0)
    return restored,dict(old_hair=old,hair=new,face=lb==FACE_ID,
        source_keep=patch==0,restore_alpha=patch,
        source_labels=lo,base_labels=lb,
        hair_preserve_guard=keep_guard,background_candidate=repair_candidate,
        background_score=bg_score,
        uncertain_new_hair=(lo==label_ids['background']) & (np.linalg.norm(np.asarray(base,dtype=np.float32)-np.asarray(original,dtype=np.float32),axis=2)>18),
        eye_core=visible_eyes,eye_keep=eye_keep,base_eye_masks=eyes,
        alignment_median_px=None,alignment_eye_max_px=None)

def single_masks(base,face_blend=JOINT_FACE_BLEND,restoration=None):
    if restoration is None:raise ValueError('ต้องมี aligned source restoration')
    face_blend=resolve_joint_settings(face_blend)['face_blend']
    if not 0<=BACKGROUND_BLEND<=1:raise ValueError('BACKGROUND_BLEND ต้องอยู่ในช่วง 0–1')
    r=restoration
    scale=min(base.size)/1024.
    target_labels=r['base_labels']
    if 'background' not in label_ids:raise ValueError('ไม่พบ background label')
    eyes=r['base_eye_masks']
    hair=r['hair']
    visible_eyes=eyes['eye_core'] & ~hair
    # Do not fill holes or union with source-face masks: bangs may cover them.
    face=((target_labels==FACE_ID) & r['face']) & ~hair
    background=(target_labels==label_ids['background']) & ~hair & ~face
    # Face/eye receive .35; hair core receives 1. Feather stays INSIDE the
    # person region so the already-good background is not regenerated.
    person=hair|face
    dist=distance_transform_edt(np.pad(person,1))[1:-1,1:-1]
    boundary=smootherstep(dist/max(1.,HAIR_BACKGROUND_FEATHER_PX*scale))
    hd=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]
    hair_weight=smootherstep(hd/max(1.,HAIR_FACE_FEATHER_PX*scale))
    alpha=(face_blend+(HAIR_DIFFUSION_BLEND-face_blend)*hair_weight)*boundary
    eye_weight=eyes['eye_keep']*(1-outward_feather(hair,max(1.,4*scale)))
    # Blend only within person; do not spill across background.
    alpha=(alpha*(1-eye_weight)+face_blend*eye_weight*boundary)*person
    alpha=alpha.astype(np.float32)
    # Bridge the hair/background boundary rather than fading to zero there.
    # A small bounded band; uncertain new hair outside the parser remains B24.
    safe_background=background & (r['background_score']>=.98) & ~r['uncertain_new_hair']
    to_hair=outside_distance(hair)
    to_bg=outside_distance(background)
    width=max(2.,SEAM_WIDTH_PX*scale)
    seam_support=(hair & (to_bg<width)) | (safe_background & (to_hair<width))
    support_distance=distance_transform_edt(np.pad(seam_support,1))[1:-1,1:-1]
    soft_support=smootherstep(support_distance/max(1.,3*scale))
    distance_to_edge=np.where(hair,to_bg,to_hair)
    seam_alpha=SEAM_BLEND*(1-smootherstep(distance_to_edge/width))*soft_support
    seam_alpha*=seam_support
    # Preserve the face/eye blend; seam does not include skin labels.
    alpha=1-(1-alpha)*(1-seam_alpha)
    alpha=alpha.astype(np.float32)
    raw=(alpha>0).astype('uint8')*255
    source_weight=(1-r['restore_alpha'])*(1-alpha)
    b24_weight=r['restore_alpha']*(1-alpha)
    assert np.allclose(source_weight+b24_weight+alpha,1,atol=1e-6)
    return dict(r,alpha=alpha,raw_mask=Image.fromarray(raw),face=face,
        background=background,eye_core=visible_eyes,eye_keep=eye_weight,
        edit_core=hair|face|background,source_weight=source_weight,b24_weight=b24_weight,
        seam_mask=(alpha>0)&(alpha<1),seam_edit_alpha=seam_alpha)

def refine_single(base,masks,seed,settings):
    if DIFFUSION_SIDE%8 or DIFFUSION_SIDE<512:raise ValueError('DIFFUSION_SIDE ต้อง >=512 และหาร 8 ลงตัว')
    image,box=padded(base,target=DIFFUSION_SIDE)
    mask,_=padded(masks['raw_mask'],target=DIFFUSION_SIDE,mask=True)
    if not np.asarray(mask).any():raise ValueError('พื้นที่แก้ว่าง')
    with torch.inference_mode():
        result=inpaint_pipe(image=image,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,
            prompt=PROMPT,negative_prompt=NEGATIVE,num_images_per_prompt=1,
            num_inference_steps=settings['steps'],strength=settings['strength'],
            guidance_scale=settings['guidance'],
            generator=torch.Generator(device=DEVICE).manual_seed(seed))
    flags=getattr(result,'nsfw_content_detected',None)
    if flags is not None and any(flags):raise ValueError('โมเดลระงับภาพผลลัพธ์')
    edited=result.images[0].convert('RGB').crop(box).resize(base.size,Image.Resampling.LANCZOS)
    if masks.get('_capture_raw',True):masks['_raw_diffusion']=edited.copy()
    return composite_protected(base,edited,masks['alpha'])


def repair_background(image, original, restoration, seed):
    """Conservative automatic mask; never selects damage from dark color alone."""
    from scipy.ndimage import binary_dilation, gaussian_filter
    r=restoration; scale=min(image.size)/1024.
    labels=r['base_labels'];source_labels=r['source_labels']
    bg_id=label_ids['background']
    target_bg=labels==bg_id
    guard=r['hair_preserve_guard']
    candidate=r['background_candidate'].copy()
    if BACKGROUND_REPAIR_EXTRA_MASK:
        with Image.open(BACKGROUND_REPAIR_EXTRA_MASK) as m:
            if m.size!=image.size:raise ValueError('extra mask ต้องมีขนาดเท่าภาพ aligned')
            manual=np.asarray(m.convert('L'))>127
        candidate |= manual & target_bg & ~guard
    if not candidate.any():
        return image.copy(),np.zeros(labels.shape,np.float32),dict(method='skipped',passes=0,
            reason='No confident background repair region; uncertain pixels preserved as B24.')
    # Feather outward around the damage, and inward near protected anatomy.
    width=max(1.,BACKGROUND_REPAIR_FEATHER_PX*scale)
    # Feather INSIDE manual support so repair cannot spill onto unmarked hair.
    inside=distance_transform_edt(np.pad(candidate,1))[1:-1,1:-1]
    alpha=smootherstep(inside/width)
    clearance=distance_transform_edt(~guard)
    alpha*=smootherstep(clearance/width)
    alpha[guard]=0
    # Source edge background samples are independent of generated black/blue defects.
    border=np.zeros(labels.shape,bool);bw=max(2,round(32*scale))
    border[:bw]=True;border[-bw:]=True;border[:,:bw]=True;border[:,-bw:]=True
    clean=(source_labels==bg_id)&border&~r['old_hair']
    samples=np.asarray(original)[clean].astype(np.float32)
    flat=False;color=None;p90=None
    if len(samples)>=100:
        color=np.median(samples,axis=0)
        errors=np.linalg.norm(samples-color,axis=1)
        p90=float(np.percentile(errors,90))
        flat=p90<=12.0
    if BACKGROUND_REPAIR_MODE not in ('manual_only','auto','inpaint','flat'):
        raise ValueError('BACKGROUND_REPAIR_MODE ต้องเป็น auto/inpaint/flat')
    use_flat=BACKGROUND_REPAIR_MODE=='flat' or (BACKGROUND_REPAIR_MODE in ('auto','manual_only') and flat)
    if use_flat and not flat:
        raise ValueError('ตรวจพื้นหลังสีเรียบไม่ผ่าน ใช้ auto หรือ inpaint')
    passes=0
    if use_flat:
        edited=Image.new('RGB',image.size,tuple(np.rint(color).astype(int)))
        method='flat_source_background'
    else:
        prompt=('Continue only the existing background, matching surrounding texture, '
                'colors and lighting, clean detailed seamless background.')
        negative='hair, person, face, ears, clothing, black fragments, new objects, text, artifacts'
        canvas,box=padded(image,target=DIFFUSION_SIDE)
        mask,_=padded(Image.fromarray((alpha>0).astype('uint8')*255),target=DIFFUSION_SIDE,mask=True)
        with torch.inference_mode():
            output=inpaint_pipe(image=canvas,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,
                prompt=prompt,negative_prompt=negative,strength=BACKGROUND_REPAIR_STRENGTH,
                num_inference_steps=BACKGROUND_REPAIR_STEPS,guidance_scale=6.5,
                generator=torch.Generator(device=DEVICE).manual_seed((int(seed)+101)%4294967296))
        flags=getattr(output,'nsfw_content_detected',None)
        if flags is not None and any(flags):raise ValueError('โมเดลระงับภาพซ่อมพื้นหลัง')
        edited=output.images[0].convert('RGB').crop(box).resize(image.size,Image.Resampling.LANCZOS)
        method='background_inpaint';passes=1
    fixed=composite_protected(image,edited,alpha)
    assert np.array_equal(np.asarray(fixed)[guard],np.asarray(image)[guard])
    assert np.array_equal(np.asarray(fixed)[alpha==0],np.asarray(image)[alpha==0])
    return fixed,alpha,dict(method=method,passes=passes,edge_background_p90=p90,
        sampled_color_rgb=None if color is None else np.rint(color).astype(int).tolist(),
        candidate_pixels=int(candidate.sum()),full_replacement_pixels=int((alpha==1).sum()),
        repair_pixels=int((alpha>0).sum()),strength=BACKGROUND_REPAIR_STRENGTH if passes else None)

# เตรียมก่อนอัปโหลด/รับคำขอ API ไม่โหลดโมเดลในคำขอแรก
PRELOAD_EYE_MODELS=False  # เว็บใช้ face segmentation; debug จะโหลด landmark เมื่อจำเป็น
if PRELOAD_EYE_MODELS:prepare_eye_models()


COLOR_LOCK_MODE = 'auto'  # auto / smoke_gray / blue / off

def validate_color_lock(mode):
    mode=str(mode).strip().lower()
    if mode not in ('auto','smoke_gray','blue','off'):
        raise ValueError('color_lock ต้องเป็น auto, smoke_gray, blue หรือ off')
    return mode

def preserve_hair_color(b24, edited, masks, mode=COLOR_LOCK_MODE):
    from skimage.color import rgb2lab,lab2rgb
    from scipy.ndimage import gaussian_filter
    mode=validate_color_lock(mode)
    hair=np.asarray(masks['hair'],bool)
    zero=np.zeros(hair.shape,np.float32)
    if mode=='off' or hair.sum()<32:
        return edited.copy(),zero,dict(requested=mode,applied='off',reason='disabled_or_small_mask')
    src=rgb2lab(np.asarray(b24.convert('RGB'))).astype(np.float32)
    dst=rgb2lab(np.asarray(edited.convert('RGB'))).astype(np.float32)
    interior=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]
    sample=hair&(interior>=max(1.,min(b24.size)/256.))
    if sample.sum()<32:sample=hair
    L=src[:,:,0][sample];ab=src[:,:,1:][sample]
    C=np.linalg.norm(ab,axis=1)
    blue_share=float(np.mean((ab[:,1]<-10)&(C>15)))
    gray_share=float(np.mean((C<10)&(L>30)&(L<85)))
    applied=mode
    if mode=='auto':
        if blue_share>=.70:applied='blue'
        elif gray_share>=.80 and float(np.median(C))<8:applied='smoke_gray'
        else:applied='off'
    info=dict(requested=mode,applied=applied,blue_share=blue_share,gray_share=gray_share,
              reference='hairfast_B24',auto_is_heuristic=True)
    if applied=='off':return edited.copy(),zero,info
    # Mask-normalized low frequencies exclude skin/background from color estimates.
    sigma=max(1.,4.*min(b24.size)/1024.)
    weight=hair.astype(np.float32)
    denom=np.maximum(gaussian_filter(weight,sigma),1e-6)
    def low(lab):
        return np.stack([gaussian_filter(lab[:,:,i]*weight,sigma)/denom for i in range(3)],axis=-1)
    low_src,low_dst=low(src),low(dst)
    corrected=dst.copy()
    # Recover chroma without copying B24's blurred luminance texture.
    corrected[:,:,1:]=.90*low_src[:,:,1:]+.10*low_dst[:,:,1:]+.20*(dst[:,:,1:]-low_dst[:,:,1:])
    luminance_gain=.80 if applied=='smoke_gray' else .35
    corrected[:,:,0]=np.clip(dst[:,:,0]+luminance_gain*(low_src[:,:,0]-low_dst[:,:,0]),0,100)
    rgb=np.rint(lab2rgb(corrected).clip(0,1)*255).astype(np.uint8)
    edge=smootherstep(interior/max(1.,HAIR_FACE_FEATHER_PX*min(b24.size)/1024.))
    alpha=edge*hair*np.asarray(masks['alpha'],np.float32)
    final=composite_protected(edited,Image.fromarray(rgb),alpha)
    changed=rgb2lab(np.asarray(final))
    info.update(luminance_gain=luminance_gain,chroma_gain=.90,
        mean_chroma_error_before=float(np.linalg.norm(dst[:,:,1:]-src[:,:,1:],axis=-1)[sample].mean()),
        mean_chroma_error_after=float(np.linalg.norm(changed[:,:,1:]-src[:,:,1:],axis=-1)[sample].mean()),
        median_L_B24=float(np.median(L)),median_L_before=float(np.median(dst[:,:,0][sample])),
        median_L_after=float(np.median(changed[:,:,0][sample])))
    assert np.array_equal(np.asarray(final)[alpha==0],np.asarray(edited)[alpha==0])
    return final,alpha,info


HAIR_DETAIL_AMOUNT=0.25  # 0 disables; subtle local contrast, not new strands

def hair_detail_finish(image,masks):
    from scipy.ndimage import gaussian_filter,minimum_filter,maximum_filter
    from skimage.color import rgb2lab,lab2rgb
    hair=np.asarray(masks['hair'],bool)
    if HAIR_DETAIL_AMOUNT==0:return image.copy(),np.zeros_like(hair,dtype=np.float32)
    if not 0<=HAIR_DETAIL_AMOUNT<=.5:raise ValueError('HAIR_DETAIL_AMOUNT ต้องอยู่ระหว่าง 0–0.5')
    lab=rgb2lab(np.asarray(image)).astype(np.float32);L=lab[:,:,0]
    sigma=max(.6,.9*min(image.size)/1024.)
    residual=L-gaussian_filter(L,sigma)
    # Threshold noise; cap contrast and clamp to local range to avoid overshoot halos.
    detail=np.sign(residual)*np.maximum(np.abs(residual)-.35,0)
    corrected=np.clip(L+np.clip(HAIR_DETAIL_AMOUNT*detail,-1.5,1.5),
                      minimum_filter(L,3),maximum_filter(L,3))
    out=lab.copy();out[:,:,0]=corrected
    distance=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]
    scale=min(image.size)/1024.
    # No sharpening near the hair silhouette, skin, background or unedited B24.
    alpha=smootherstep((distance-max(2.,8*scale))/max(1.,4*scale))*masks['alpha']
    alpha*=hair
    rgb=np.rint(lab2rgb(out).clip(0,1)*255).astype(np.uint8)
    final=composite_protected(image,Image.fromarray(rgb),alpha)
    assert np.array_equal(np.asarray(final)[alpha==0],np.asarray(image)[alpha==0])
    return final,alpha


DETAIL_GAIN=.65
DETAIL_MAX_RGB=5.0

def detail_only_masks(base,face_blend):
    labels=segment_labels(base)  # exactly one parser pass
    hair=labels==HAIR_ID;face=labels==FACE_ID
    if not hair.any():raise ValueError('ตรวจไม่พบผม')
    def inside(mask,width):
        distance=distance_transform_edt(np.pad(mask,1))[1:-1,1:-1]
        return smootherstep(distance/max(1.,width*min(base.size)/1024.))
    # No hair silhouette displacement: taper inside each edit region only.
    hair_alpha=inside(hair,4.)
    face_alpha=inside(face,6.)*float(face_blend)
    alpha=np.maximum(hair_alpha,face_alpha).astype(np.float32)
    return dict(hair=hair,face=face,alpha=alpha,
        raw_mask=Image.fromarray((alpha>0).astype('uint8')*255))

def transfer_luminance_detail(b24,proposal,alpha):
    from scipy.ndimage import gaussian_filter
    source=np.asarray(b24.convert('RGB'),dtype=np.float32)
    generated=np.asarray(proposal.convert('RGB'),dtype=np.float32)
    if source.shape!=generated.shape:raise ValueError('ภาพต้องมีขนาดเท่ากัน')
    coeff=np.array([.299,.587,.114],np.float32)
    y0=source@coeff;y1=generated@coeff
    sigma=max(.8,1.2*min(b24.size)/1024.)
    detail0=y0-gaussian_filter(y0,sigma)
    detail1=y1-gaussian_filter(y1,sigma)
    # Preserve B24 coarse structure/color; use only bounded high-frequency delta.
    delta=np.clip(DETAIL_GAIN*(detail1-detail0),-DETAIL_MAX_RGB,DETAIL_MAX_RGB)*alpha
    # Equal RGB increments preserve channel differences; no individual-channel clipping.
    delta=np.clip(delta,-source.min(axis=2),255-source.max(axis=2))
    shift=np.rint(delta).astype(np.int16)
    out=(source.astype(np.int16)+shift[:,:,None]).astype(np.uint8)
    out[alpha==0]=source.astype(np.uint8)[alpha==0]
    return Image.fromarray(out)

def refine_b24_details(base,masks,seed,settings):
    canvas,box=padded(base,target=DIFFUSION_SIDE)
    mask,_=padded(masks['raw_mask'],target=DIFFUSION_SIDE,mask=True)
    with torch.inference_mode():
        result=inpaint_pipe(image=canvas,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,
            prompt='Same portrait, same hairstyle and hair color, fine natural hair detail, clear eyes, natural skin texture, realistic photograph.',
            negative_prompt='changed hairstyle, changed hair color, changed face, waxy skin, blur, halos, noise',
            num_images_per_prompt=1,strength=settings['strength'],num_inference_steps=settings['steps'],
            guidance_scale=settings['guidance'],generator=torch.Generator(device=DEVICE).manual_seed(int(seed)))
    flags=getattr(result,'nsfw_content_detected',None)
    if flags is not None and any(flags):raise ValueError('โมเดลระงับภาพผลลัพธ์')
    proposal=result.images[0].convert('RGB').crop(box).resize(base.size,Image.Resampling.LANCZOS)
    return proposal


RAW_BG_GAP_PX=4.0  # pixels at 1024, scaled with output size
RAW_BG_FEATHER_PX=32.0
RAW_BG_DETAIL_AMOUNT=.45

def finish_raw_background(raw,b24_masks,labels=None):
    from scipy.ndimage import gaussian_filter
    if RAW_BG_GAP_PX<0 or RAW_BG_FEATHER_PX<=0 or not 0<=RAW_BG_DETAIL_AMOUNT<=1:
        raise ValueError('ตรวจค่า RAW_BG_GAP_PX / FEATHER / DETAIL_AMOUNT')
    # Hair may move in raw diffusion: protect both B24 and raw hair.
    if labels is None:labels=segment_labels(raw)
    if 'background' not in label_ids:raise ValueError('ไม่พบ background label')
    bg=labels==label_ids['background']
    hair=(labels==HAIR_ID)|b24_masks['hair']
    scale=min(raw.size)/1024.
    hair_distance=outside_distance(hair)
    gap=RAW_BG_GAP_PX*scale;feather=max(1.,RAW_BG_FEATHER_PX*scale)
    # Stay inside background; protect skin, ears, neck and clothes too.
    bg_distance=distance_transform_edt(np.pad(bg,1))[1:-1,1:-1]
    # One continuous ramp from the nearest protected boundary.
    # Feather the adjustment weight, never the output image.
    safe_distance=np.minimum(hair_distance,bg_distance)
    alpha=(smootherstep((safe_distance-gap)/feather)*bg).astype(np.float32)
    rgb=np.asarray(raw.convert('RGB'),dtype=np.float32)
    luminance=rgb@np.array([.299,.587,.114],np.float32)
    detail=luminance-gaussian_filter(luminance,max(.8,1.2*scale))
    detail=np.sign(detail)*np.maximum(np.abs(detail)-.5,0.)
    delta=np.clip(RAW_BG_DETAIL_AMOUNT*detail,-4.,4.)*alpha
    delta=np.clip(delta,-rgb.min(axis=2),255-rgb.max(axis=2))
    out=(rgb.astype(np.int16)+np.rint(delta).astype(np.int16)[:,:,None]).astype(np.uint8)
    protected=(safe_distance<=gap)|~bg
    assert not alpha[protected].any()
    assert np.array_equal(out[protected],rgb.astype(np.uint8)[protected])
    return Image.fromarray(out),alpha,dict(method='raw_background_smooth_gap_detail',
        gap_px_at_1024=RAW_BG_GAP_PX,feather_px_at_1024=RAW_BG_FEATHER_PX,
        amount=RAW_BG_DETAIL_AMOUNT,protected_raw_max_diff=0,extra_diffusion_passes=0)


# Lightweight native-resolution finish. No extra neural model / diffusion pass.
# Defaults keep the ORIGINAL diffusion prompt and sampling parameters unchanged.
NATIVE_SHARP_ENABLED = True
NATIVE_HAIR_AMOUNT = 0.65
NATIVE_FACE_AMOUNT = 0.25
NATIVE_SHARP_MAX_DELTA = 5.0  # maximum equal RGB shift, in 8-bit levels
NATIVE_SHARP_THRESHOLD = 0.8  # suppress amplification of very small residuals

def finish_native_sharp(image, labels):
    from scipy.ndimage import gaussian_filter
    if not (0 <= NATIVE_HAIR_AMOUNT <= 1.5 and 0 <= NATIVE_FACE_AMOUNT <= .75
            and 0 <= NATIVE_SHARP_MAX_DELTA <= 10 and NATIVE_SHARP_THRESHOLD >= 0):
        raise ValueError('ตรวจค่า NATIVE_SHARP: hair 0–1.5, face 0–.75, delta 0–10')
    rgb=np.asarray(image.convert('RGB'),dtype=np.float32)
    if labels.shape != rgb.shape[:2]:raise ValueError('ขนาดมาสก์ไม่ตรงกับภาพ')
    info=dict(method='bounded_native_luminance_sharpen',enabled=bool(NATIVE_SHARP_ENABLED),
              hair_amount=NATIVE_HAIR_AMOUNT,face_amount=NATIVE_FACE_AMOUNT,
              max_delta=NATIVE_SHARP_MAX_DELTA,threshold=NATIVE_SHARP_THRESHOLD,
              output_size=list(image.size),extra_diffusion_passes=0)
    if not NATIVE_SHARP_ENABLED:
        return image.copy(),dict(info,observed_max_delta=0)
    scale=min(image.size)/1024.
    def inward(region, width):
        dist=distance_transform_edt(np.pad(region,1))[1:-1,1:-1]
        # Leave outermost pixel unchanged; taper inward, never blur the image edge.
        return smootherstep((dist-1.)/max(1.,width*scale))*region
    weight=(NATIVE_HAIR_AMOUNT*inward(labels==HAIR_ID,3.)+
            NATIVE_FACE_AMOUNT*inward(labels==FACE_ID,5.)).astype(np.float32)
    lum=rgb@np.array([.299,.587,.114],np.float32)
    fine=lum-gaussian_filter(lum,max(.55,.75*scale))
    medium=lum-gaussian_filter(lum,max(.8,1.35*scale))
    residual=.75*fine+.25*medium
    residual=np.sign(residual)*np.maximum(np.abs(residual)-NATIVE_SHARP_THRESHOLD,0.)
    delta=np.clip(weight*residual,-NATIVE_SHARP_MAX_DELTA,NATIVE_SHARP_MAX_DELTA)
    # Equal-channel increments preserve RGB channel differences without clipping.
    delta=np.clip(delta,-rgb.min(axis=2),255-rgb.max(axis=2))
    shift=np.rint(delta).astype(np.int16)
    out=(rgb.astype(np.int16)+shift[:,:,None]).astype(np.uint8)
    return Image.fromarray(out),dict(info,observed_max_delta=int(np.abs(shift).max()))


## 7. สร้างผลและบันทึกภาพ / masks / metadata


In [ ]:
def run_single(face_path,shape_path,color_path,seed=DEFAULT_SEED,face_blend=JOINT_FACE_BLEND,
               strength=None,steps=None,guidance=None,color_lock=COLOR_LOCK_MODE):
    color_lock=validate_color_lock(color_lock)
    settings=resolve_inpaint_settings(strength,steps,guidance)
    resolve_joint_settings(face_blend)
    if not 0<=int(seed)<=4294967295:raise ValueError('seed ไม่ถูกต้อง')
    queued_at=time.perf_counter()
    with GPU_LOCK:
        queue_seconds=time.perf_counter()-queued_at
        started=time.perf_counter();run_id=uuid.uuid4().hex
        folder=RESULTS_ROOT/run_id;folder.mkdir()
        timings={}
        def stamp():
            if torch.cuda.is_available():torch.cuda.synchronize()
            return time.perf_counter()
        last_stamp=stamp()
        def mark(name):
            nonlocal last_stamp
            now=stamp();timings[name]=round(now-last_stamp,4);last_stamp=now
        swapped=hair_fast.swap(str(face_path),str(shape_path),str(color_path),align=True,seed=int(seed))
        mark('hairfast_B24')
        if not isinstance(swapped,tuple) or len(swapped)!=4:raise RuntimeError('swap return format ไม่ตรงรุ่น')
        base=tensor_to_pil(swapped[0]);base.save(folder/'hairfast_B24.png')
        for name,value in zip(('face_aligned','shape_reference','color_reference'),swapped[1:]):
            tensor_to_pil(value).save(folder/(name+'.png'))
        original=tensor_to_pil(swapped[1])
        mark('save_inputs')
        restored,restoration=prepare_face_base(base,original)
        mark('segmentation_and_landmarks')
        restored.save(folder/'restored_face_background.png')
        mark('save_base')
        masks=single_masks(restored,face_blend,restoration)
        mark('blend_masks')
        final=refine_single(restored,masks,int(seed),settings)
        mark('diffusion_hair_face')
        masks.pop('_raw_diffusion').save(folder/'diffusion_raw_before_composite.png')
        final.save(folder/'after_diffusion_composite.png')
        mark('save_diffusion_diagnostics')
        before_color_lock=final
        before_color_lock.save(folder/'before_color_lock.png')
        mark('save_before_color_lock')
        final,color_alpha,color_info=preserve_hair_color(base,final,masks,color_lock)
        mark('hair_color_lock')
        Image.fromarray(np.rint(color_alpha*255).astype('uint8')).save(folder/'color_lock_alpha.png')
        final.save(folder/'after_color_lock.png')
        final,detail_alpha=hair_detail_finish(final,masks)
        mark('hair_detail_finish')
        final.save(folder/'after_hair_detail.png')
        Image.fromarray(np.rint(detail_alpha*255).astype('uint8')).save(folder/'hair_detail_alpha.png')
        before_repair=final
        before_repair.save(folder/'before_background_repair.png')
        mark('save_before_repair')
        final,bg_alpha,bg_info=repair_background(before_repair,original,restoration,int(seed))
        mark('background_repair')
        assert not np.any(bg_alpha[restoration['hair_preserve_guard']]), 'Repair overlaps preserved hair'
        keep_only=restoration['hair_preserve_guard']&(masks['alpha']==0)
        assert np.array_equal(np.asarray(final)[keep_only],np.asarray(base)[keep_only])
        Image.fromarray(np.rint(bg_alpha*255).astype('uint8')).save(folder/'background_repair_alpha.png')
        overlay=np.asarray(before_repair).astype(np.float32).copy()
        marked=bg_alpha>0
        overlay[marked]=overlay[marked]*.55+np.array([255,40,40])*.45
        Image.fromarray(np.rint(overlay).astype('uint8')).save(folder/'background_repair_overlay.png')
        preserved_b24=(masks['alpha']==0)&(bg_alpha==0)
        b24_difference=np.abs(np.asarray(final).astype('int16')-np.asarray(base).astype('int16'))
        preserved_b24_max=int(b24_difference[preserved_b24].max()) if preserved_b24.any() else 0
        assert preserved_b24_max==0,'Unedited B24 pixels must remain identical'
        Image.fromarray(preserved_b24.astype('uint8')*255).save(folder/'preserved_B24_mask.png')
        final.save(folder/'result.png')
        comparison=Image.new('RGB',(final.width*2,final.height))
        comparison.paste(before_repair,(0,0));comparison.paste(final,(final.width,0))
        comparison.save(folder/'background_before_after.png')
        protected_original=masks['source_keep'] & (masks['alpha']==0) & (bg_alpha==0)
        source_diff=np.abs(np.asarray(final).astype('int16')-np.asarray(original).astype('int16'))
        if protected_original.any():assert int(source_diff[protected_original].max())==0
        Image.fromarray(protected_original.astype('uint8')*255).save(folder/'protected_original_mask.png')
        for name in ('hair','face','eye_core','eye_keep','alpha','restore_alpha',
                     'source_weight','b24_weight','seam_mask','edit_core','background','hair_preserve_guard','background_candidate','seam_edit_alpha'):
            Image.fromarray(np.rint(masks[name]*255).astype('uint8')).save(folder/(name+'_mask.png'))
        masks['raw_mask'].save(folder/'diffusion_mask.png')
        from skimage.color import rgb2lab
        a=np.asarray(restored);b=np.asarray(final)
        diff=np.abs(b.astype('int16')-a.astype('int16'))
        lab0=rgb2lab(a);lab1=rgb2lab(b);h=masks['hair'];outside=(masks['alpha']==0)&(bg_alpha==0)
        metrics=dict(preserved_B24_max_diff=preserved_b24_max,eye_max_diff=int(diff[masks['eye_core']].max()) if masks['eye_core'].any() else 0,
            outside_edit_max_diff=int(diff[outside].max()) if outside.any() else 0,
            hair_mean_pixel_deltaE76_vs_restored=float(np.linalg.norm(lab1-lab0,axis=-1)[h].mean()),
            hair_mean_abs_L_change=float(np.abs(lab1[:,:,0]-lab0[:,:,0])[h].mean()),
            seconds=time.perf_counter()-started)
        assert metrics['outside_edit_max_diff']==0
        metadata=dict(experiment_id=run_id,mode='B24_keep_hair_confident_bg_repair',result_key='result',
            checkpoint=str(CHECKPOINT),checkpoint_sha256=EXPECTED_SHA256,epoch=24,trial='B',
            mixing=.95,shape_rotation=False,color_rotation=True,seed=int(seed),face_blend=float(face_blend),face_blend_requested=float(face_blend),
            boundary_blending={'method':'regional_blend_with_bounded_seam',
                'seam_width_px_at_1024':SEAM_WIDTH_PX,'seam_blend':SEAM_BLEND,
                'hair_diffusion_blend':HAIR_DIFFUSION_BLEND,
                'restore_px_at_1024':RESTORE_FEATHER_PX,
                'hair_face_px_at_1024':HAIR_FACE_FEATHER_PX,
                'hair_background_px_at_1024':HAIR_BACKGROUND_FEATHER_PX,
                'label_transition_px_at_1024':LABEL_TRANSITION_PX},
            hair_detail_amount=HAIR_DETAIL_AMOUNT,composite_base='full_B24',face_eye_diffusion_blend=float(face_blend),background_blend=BACKGROUND_BLEND,diffusion_side=DIFFUSION_SIDE,alignment_median_px=masks['alignment_median_px'],
            prompt=PROMPT,negative_prompt=NEGATIVE,diffusion_passes=1+bg_info['passes'],background_repair=bg_info,color_lock=color_info,settings=settings,
            metrics=metrics,environment=ENVIRONMENT,
            input_sha256={k:sha_file(p) for k,p in zip(('face','shape','color'),(face_path,shape_path,color_path))})
        mark('save_artifacts_and_metrics')
        archive=Path(shutil.make_archive(str(folder),'zip',folder))
        mark('zip_archive')
        timings['total_processing']=round(time.perf_counter()-started,4)
        timings['queue_wait']=round(queue_seconds,4)
        metadata['timings_seconds']=timings
        metadata['metrics']['seconds']=timings['total_processing']
        timing_path=folder/'timings.json'
        timing_path.write_text(json.dumps(timings,indent=2))
        metadata_path=folder/'metadata.json'
        metadata_path.write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
        # Add once after measuring archive generation; no duplicate ZIP entries.
        with zipfile.ZipFile(archive,'a',compression=zipfile.ZIP_DEFLATED) as z:
            z.write(timing_path,'timings.json');z.write(metadata_path,'metadata.json')
        print('เวลาแต่ละขั้น (วินาที):')
        for name,seconds in timings.items():print(f'  {name}: {seconds:.2f}')
        return dict(image=final,folder=folder,zip_path=archive,metadata=metadata)


def run_detail_core(inputs=None,before=None,seed=3407,strength=.15,steps=30,guidance=2.5,face_blend=.35):
    # One shared implementation for the comparison cell, upload test and web.
    settings=resolve_inpaint_settings(strength,steps,guidance)
    resolve_joint_settings(face_blend)
    if not 0<=int(seed)<=4294967295:raise ValueError('seed ไม่ถูกต้อง')
    timings={}
    def clock():
        if torch.cuda.is_available():torch.cuda.synchronize()
        return time.perf_counter()
    queued=time.perf_counter()
    with GPU_LOCK:
        timings['queue_wait']=time.perf_counter()-queued
        start=clock()
        if before is None:
            if inputs is None or len(inputs)!=3:raise ValueError('ต้องมีหน้า/ทรง/สี 3 ภาพ')
            t=clock()
            swapped=hair_fast.swap(*(str(v) for v in inputs),align=True,seed=int(seed))
            if not isinstance(swapped,tuple) or len(swapped)!=4:raise RuntimeError('swap format ไม่ตรงรุ่น')
            before=tensor_to_pil(swapped[0]);timings['B24']=clock()-t
            del swapped
        else:
            before=before.convert('RGB').copy();timings['B24']=0.
        t=clock();masks=detail_only_masks(before,face_blend)
        timings['mask']=clock()-t
        t=clock();raw=refine_b24_details(before,masks,seed,settings)
        timings['diffusion']=clock()-t
        t=clock();raw_labels=segment_labels(raw)
        before_sharp,bg_alpha,bg_info=finish_raw_background(raw,masks,labels=raw_labels)
        timings['raw_background_finish']=clock()-t
        t=clock();after,sharp_info=finish_native_sharp(before_sharp,raw_labels)
        timings['native_sharp']=clock()-t
        timings['total_processing']=clock()-start
    return dict(before=before,raw_diffusion=raw,before_sharp=before_sharp,after=after,native_sharp=sharp_info,sharp_labels=raw_labels,masks=masks,background_alpha=bg_alpha,background_info=bg_info,settings=settings,
                seed=int(seed),timings_seconds={k:round(v,4) for k,v in timings.items()})

def run_single_fast(face_path,shape_path,color_path,seed=3407,face_blend=.35,
                    strength=.15,steps=30,guidance=2.5,color_lock='auto'):
    validate_color_lock(color_lock)
    result=run_detail_core([face_path,shape_path,color_path],seed=seed,
        strength=.15 if strength is None else strength,steps=30 if steps is None else steps,
        guidance=2.5 if guidance is None else guidance,face_blend=face_blend)
    timings=result['timings_seconds']
    metadata=dict(experiment_id=uuid.uuid4().hex,mode='diffusion_native_sharp',result_key='result',
        trial='B',epoch=24,mixing=.95,shape_rotation=False,color_rotation=True,
        seed=int(seed),face_blend=float(face_blend),diffusion_side=DIFFUSION_SIDE,
        settings=result['settings'],output_base='raw_diffusion',face_blend_applied=False,
        color_lock={'applied':'none_raw_selected','legacy_requested':color_lock},
        background_repair={**result['background_info'],'passes':0},diffusion_passes=1,
        native_sharp=result['native_sharp'],
        timings_seconds=timings,metrics={'seconds':timings['total_processing']})
    print('B24 shared comparison core:',timings)
    return dict(image=result['after'],metadata=metadata)

COMPARE_WEB_PROFILE = {'strength':0.15,'steps':30,'guidance':2.5,'face_blend':0.35}
COMPARE_WEB_BUILD = 'looklab-native-sharp-v1'

def run_compare_profile(face_path,shape_path,color_path,seed=3407,
                        strength=None,steps=None,guidance=None,face_blend=None,color_lock='auto'):
    # Fixed profile selected for web parity; legacy UI settings are recorded, not applied.
    requested=dict(strength=strength,steps=steps,guidance=guidance,face_blend=face_blend)
    print(COMPARE_WEB_BUILD,'effective:',COMPARE_WEB_PROFILE,'side:',DIFFUSION_SIDE,'seed:',seed)
    result=run_single_fast(face_path,shape_path,color_path,seed=seed,
                          color_lock=color_lock,**COMPARE_WEB_PROFILE)
    result['metadata']['build']=COMPARE_WEB_BUILD
    result['metadata']['profile']='same_as_compare_cell_defaults'
    result['metadata']['requested_legacy_settings']=requested
    return result


## 8. อัปโหลดรูปหน้า → ทรงผม → สีผม และดาวน์โหลด ZIP


In [ ]:
RUN_UPLOAD_TEST= True
DEBUG_EXPORT=False  # True เท่านั้นจึงบันทึก diagnostics และ ZIP
COLOR_LOCK_CHOICE='auto'  # เลือก smoke_gray / blue / off ได้ตรงนี้
if RUN_UPLOAD_TEST:
    from google.colab import files
    from IPython.display import display
    import tempfile
    input_dir=Path(tempfile.mkdtemp(prefix='hfg_upload_'))
    paths=[]
    for role,label in [('face','รูปหน้า'),('shape','รูปทรงผม'),('color','รูปสีผม')]:
        print('อัปโหลด '+label+' 1 ภาพ')
        uploaded=files.upload()
        if len(uploaded)!=1:raise ValueError('เลือกครั้งละหนึ่งภาพ')
        with Image.open(io.BytesIO(next(iter(uploaded.values())))) as im:
            path=input_dir/(role+'.png');ImageOps.exif_transpose(im).convert('RGB').save(path)
        paths.append(path)
    single_result=(run_single if DEBUG_EXPORT else run_compare_profile)(*paths,color_lock=COLOR_LOCK_CHOICE)
    display(single_result['image'])
    print('Hair color lock:',single_result['metadata']['color_lock'])
    print('Background repair:',single_result['metadata']['background_repair'])
    if DEBUG_EXPORT:display(Image.open(single_result['folder']/'background_repair_overlay.png'))
    print(single_result['metadata']['metrics'])
    if DEBUG_EXPORT:files.download(str(single_result['zip_path']))

    shutil.rmtree(input_dir,ignore_errors=True)


## 9. API สำหรับเว็บ — ผลเดียว
POST `/generate` หรือ `/jobs` ด้วย `face_image`, `shape_image`, `color_image`, `seed`, `strength`, `steps`, `guidance`, `face_blend`.
`/generate` คืน `images.result` เป็น PNG base64 เพียงหนึ่งภาพ (ไม่มี data-URL prefix). ใช้ `/jobs` หากเว็บรอคำขอนาน: POST → GET `/jobs/{job_id}` → GET `/jobs/{job_id}/result`.

เว็บต้องเอาปุ่ม 7 ภาพและตัวเลือกโหมดเก่าออก เหลือ Joint ควบคุมหน้าแยกตาเดียว โค้ด frontend ไม่ได้แนบมาจึงไม่ได้แก้หน้าเว็บในไฟล์นี้ ใช้ `experiment_id`, `result_key=result`, `mode=joint_controlled_eyes` ส่ง feedback ตามเดิม

ค่าเริ่มต้นของ Colab และ API อ่านจาก INPAINT_STRENGTH / INPAINT_STEPS / INPAINT_GUIDANCE ในข้อ 6 ชุดเดียวกัน หากเว็บส่งค่ามาชัดเจนจะใช้ค่าจากเว็บ ตรวจ settings ใน metadata.json เพื่อดูค่าที่ใช้จริง


In [ ]:
from fastapi import FastAPI,File,UploadFile,Form,HTTPException
from fastapi.responses import FileResponse, Response
from fastapi.middleware.cors import CORSMiddleware
from concurrent.futures import ThreadPoolExecutor
import base64, tempfile
app=FastAPI(title='HairFastGAN B24 + Face 0.35 + Background')
app.add_middleware(CORSMiddleware,allow_origins=['*'],allow_methods=['*'],allow_headers=['*'])
def save_inputs(uploads):
    folder=Path(tempfile.mkdtemp(prefix='hfg_single_'));paths=[]
    try:
        for role,upload in zip(('face','shape','color'),uploads):
            data=upload.file.read(20*1024*1024+1)
            if len(data)>20*1024*1024:raise ValueError('ไฟล์เกิน 20 MB')
            with Image.open(io.BytesIO(data)) as im:
                p=folder/(role+'.png');ImageOps.exif_transpose(im).convert('RGB').save(p);paths.append(p)
        return folder,paths
    except Exception:
        shutil.rmtree(folder,ignore_errors=True);raise

def result_payload(result):
    buf=io.BytesIO();result['image'].save(buf,format='PNG')
    return {**result['metadata'],'images':{'result':base64.b64encode(buf.getvalue()).decode()},'image_count':1}
@app.get('/health')
def health():return {'status':'ok','trial':'B','epoch':24,'image_count':1,'build':COMPARE_WEB_BUILD}
@app.get('/settings')
def api_settings():
    return {'modes':['raw_diffusion_guarded_background'],'image_count':1,
            'build':COMPARE_WEB_BUILD,'fixed_profile':True,
            'defaults':{**COMPARE_WEB_PROFILE,'seed':3407,'color_lock':'auto'},
            'note':'Web uses the comparison defaults; legacy strength/steps/guidance/face_blend fields are recorded but overridden.'}
@app.post('/generate')
def generate(face_image:UploadFile=File(...),shape_image:UploadFile=File(...),color_image:UploadFile=File(...),
             seed:int=Form(DEFAULT_SEED),strength:float=Form(None),steps:int=Form(None),guidance:float=Form(None),face_blend:float=Form(JOINT_FACE_BLEND),color_lock:str=Form(COLOR_LOCK_MODE)):
    folder,paths=save_inputs((face_image,shape_image,color_image))
    try:return result_payload(run_compare_profile(*paths,seed=seed,strength=strength,steps=steps,guidance=guidance,face_blend=face_blend,color_lock=color_lock))
    except ValueError as e:raise HTTPException(422,str(e))
    finally:shutil.rmtree(folder,ignore_errors=True)
_POOL=ThreadPoolExecutor(max_workers=1);_JOBS={};_JOBS_LOCK=threading.Lock()
def worker(job,folder,paths,settings):
    _JOBS[job]['status']='running'
    try:
        result=run_compare_profile(*paths,**settings)
        _JOBS[job].update(status='complete',result=result)
    except Exception as e:_JOBS[job].update(status='error',error=str(e))
    finally:shutil.rmtree(folder,ignore_errors=True)
@app.post('/jobs')
def submit(face_image:UploadFile=File(...),shape_image:UploadFile=File(...),color_image:UploadFile=File(...),
           seed:int=Form(DEFAULT_SEED),strength:float=Form(None),steps:int=Form(None),guidance:float=Form(None),face_blend:float=Form(JOINT_FACE_BLEND),color_lock:str=Form(COLOR_LOCK_MODE)):
    try:validate_color_lock(color_lock)
    except ValueError as e:raise HTTPException(422,str(e))
    with _JOBS_LOCK:
        if sum(j['status'] in ('queued','running') for j in _JOBS.values())>=4:raise HTTPException(429,'คิวเต็ม')
        folder,paths=save_inputs((face_image,shape_image,color_image));job=uuid.uuid4().hex
        _JOBS[job]={'status':'queued'}
    _POOL.submit(worker,job,folder,paths,dict(seed=seed,strength=strength,steps=steps,guidance=guidance,face_blend=face_blend,color_lock=color_lock))
    return {'job_id':job,'status':'queued'}
def get_job(job_id):
    if job_id not in _JOBS:raise HTTPException(404,'ไม่พบงาน')
    return _JOBS[job_id]
@app.get('/jobs/{job_id}')
def job_status(job_id:str):return {k:v for k,v in get_job(job_id).items() if k!='result'}
@app.get('/jobs/{job_id}/result')
def job_result(job_id:str):
    j=get_job(job_id)
    if j['status']!='complete':raise HTTPException(409,j['status'])
    return result_payload(j['result'])
@app.get('/jobs/{job_id}/download')
def job_download(job_id:str):
    j=get_job(job_id)
    if j['status']!='complete':raise HTTPException(409,j['status'])
    # ZIP is made only if a user explicitly requests this download endpoint.
    result=j['result'];stream=io.BytesIO();png=io.BytesIO()
    result['image'].save(png,format='PNG')
    with zipfile.ZipFile(stream,'w',compression=zipfile.ZIP_STORED) as z:
        z.writestr('result.png',png.getvalue())
        z.writestr('metadata.json',json.dumps(result['metadata'],ensure_ascii=False,indent=2))
    return Response(stream.getvalue(),media_type='application/zip',
        headers={'Content-Disposition':'attachment; filename="result.zip"'})


## 10. รับคะแนน feedback — คงโค้ดเดิม
ใช้ Colab Secrets `FEEDBACK_APPS_SCRIPT_URL` และ `FEEDBACK_SHARED_SECRET` ตามเดิม

In [ ]:
# Run this cell AFTER the FastAPI definitions and BEFORE starting uvicorn/ngrok.
# Does not change model inference or existing endpoints.
import base64
import hashlib
import io
import json
import math
import re
import threading
from urllib.parse import urlparse
import requests
from fastapi import File, Form, HTTPException, UploadFile
from PIL import Image, ImageOps

_FEEDBACK_SLOTS = threading.BoundedSemaphore(2)
_FEEDBACK_ID = re.compile(r'^[A-Za-z0-9_-]{8,100}$')
_FEEDBACK_IMAGE_MAX = 4 * 1024 * 1024
_FEEDBACK_AGE_GROUPS = ('ต่ำกว่า 18 ปี', '18–24 ปี', '25–34 ปี', '35–44 ปี', '45–54 ปี', '55 ปีขึ้นไป')


def validate_feedback(data):
    if not isinstance(data, dict):
        raise ValueError('metadata ต้องเป็น JSON object')
    allowed = {'experiment_id','result_key','experiment_type','hairstyle_id','hairstyle_name',
               'hair_color_id','hair_color_name','mode','seed','source_endpoint',
               'user_wait_seconds','hairstyle_rating','hair_color_rating',
               'face_similarity_rating','waiting_satisfaction_rating','suggestion',
               'image_storage_consent','overall_satisfaction_rating','age_group','website_satisfaction_rating'}
    if set(data) - allowed:
        raise ValueError('พบชื่อข้อมูลที่ไม่รองรับ')
    out = dict(data)
    if out.get('age_group') not in _FEEDBACK_AGE_GROUPS:
        raise ValueError('กรุณาเลือกช่วงอายุจาก 6 ตัวเลือกที่กำหนด')
    if not isinstance(out.get('experiment_id'), str) or not _FEEDBACK_ID.fullmatch(out['experiment_id']):
        raise ValueError('experiment_id ต้องยาว 8–100 ตัว ใช้ตัวอักษรอังกฤษ ตัวเลข - หรือ _')
    for name, limit in [('result_key',120),('hairstyle_id',100),('hairstyle_name',200),
                        ('hair_color_id',100),('hair_color_name',200),('mode',120),
                        ('source_endpoint',100),('suggestion',2000)]:
        value = out.get(name, '')
        if not isinstance(value, str) or len(value) > limit:
            raise ValueError(f'{name} ต้องเป็นข้อความไม่เกิน {limit} ตัวอักษร')
        out[name] = value.strip()
    if not out['result_key']:
        raise ValueError('ต้องระบุ result_key ของภาพที่กำลังประเมิน')
    kind = out.get('experiment_type')
    if kind not in ('hairstyle','color','both'):
        raise ValueError('experiment_type ต้องเป็น hairstyle, color หรือ both')
    if kind in ('hairstyle','both') and not out['hairstyle_name']:
        raise ValueError('ต้องระบุชื่อทรงผมที่ใช้สร้างผลลัพธ์')
    if kind in ('color','both') and not out['hair_color_name']:
        raise ValueError('ต้องระบุชื่อสีผมที่ใช้สร้างผลลัพธ์')
    required = ['face_similarity_rating','waiting_satisfaction_rating','overall_satisfaction_rating','website_satisfaction_rating']
    if kind in ('hairstyle','both'): required.append('hairstyle_rating')
    if kind in ('color','both'): required.append('hair_color_rating')
    for name in ('hairstyle_rating','hair_color_rating','face_similarity_rating','waiting_satisfaction_rating','overall_satisfaction_rating','website_satisfaction_rating'):
        value = out.get(name)
        if name in required:
            if type(value) is not int or not 1 <= value <= 5:
                raise ValueError(f'{name} ต้องเป็นคะแนนจำนวนเต็ม 1–5')
        elif value is not None:
            raise ValueError(f'{name} ไม่เกี่ยวข้องกับการทดลองนี้ ให้ส่ง null หรือไม่ส่ง')
        out[name] = value
    if type(out.get('image_storage_consent')) is not bool:
        raise ValueError('image_storage_consent ต้องเป็น true หรือ false')
    wait = out.get('user_wait_seconds')
    if type(wait) not in (int,float) or not math.isfinite(wait) or not 0 <= wait <= 86400:
        raise ValueError('user_wait_seconds ต้องเป็นตัวเลข 0–86400')
    out['user_wait_seconds'] = round(float(wait), 3)
    seed = out.get('seed')
    if seed is not None and (type(seed) is not int or not 0 <= seed <= 4294967295):
        raise ValueError('seed ไม่ถูกต้อง')
    out['seed'] = seed
    return out


def feedback_settings():
    from google.colab import userdata
    try:
        url = userdata.get('FEEDBACK_APPS_SCRIPT_URL').strip()
        secret = userdata.get('FEEDBACK_SHARED_SECRET')
    except Exception:
        raise HTTPException(503,'เพิ่ม FEEDBACK_APPS_SCRIPT_URL และ FEEDBACK_SHARED_SECRET ใน Colab Secrets แล้วเปิด Notebook access')
    parsed = urlparse(url)
    if parsed.scheme != 'https' or parsed.netloc != 'script.google.com' or not re.fullmatch(r'/macros/s/[A-Za-z0-9_-]+/exec', parsed.path) or parsed.query:
        raise HTTPException(503,'FEEDBACK_APPS_SCRIPT_URL ต้องเป็น URL Web app ของ Apps Script ที่ลงท้าย /exec')
    if len(secret) < 32:
        raise HTTPException(503,'FEEDBACK_SHARED_SECRET ต้องยาวอย่างน้อย 32 ตัวอักษร')
    return url, secret


def feedback_image(upload):
    if upload is None:
        raise HTTPException(422,'เมื่อยินยอมเก็บภาพ ต้องส่ง original_image และ result_image')
    raw = upload.file.read(12 * 1024 * 1024 + 1)
    if len(raw) > 12 * 1024 * 1024:
        raise HTTPException(413,'รูปใหญ่เกิน 12 MB')
    try:
        with Image.open(io.BytesIO(raw)) as image:
            if image.format not in ('PNG','JPEG','WEBP') or image.width * image.height > 16_000_000:
                raise HTTPException(422,'รองรับ PNG/JPG/WEBP ไม่เกิน 16 ล้านพิกเซล')
            image = ImageOps.exif_transpose(image).convert('RGB')
            # Keep small images intact; shrink oversized PNGs without changing aspect ratio.
            source_size = image.size
            for attempt in range(20):
                buffer = io.BytesIO()
                image.save(buffer, format='PNG')
                encoded = buffer.getvalue()
                if len(encoded) <= _FEEDBACK_IMAGE_MAX:
                    return {'base64':base64.b64encode(encoded).decode('ascii'),
                            'mime_type':'image/png','source_size':list(source_size),
                            'saved_size':list(image.size),'resized':image.size != source_size}
                if max(image.size) <= 256:
                    break
                scale = min(0.8, 2048 / max(image.size))
                image = image.resize((max(1,round(image.width*scale)),
                                      max(1,round(image.height*scale))), Image.Resampling.LANCZOS)
            raise HTTPException(413,'ไม่สามารถย่อรูปให้ต่ำกว่า 4 MB ได้ กรุณาเลือกไฟล์ภาพใหม่')
    except HTTPException:
        raise
    except Exception:
        raise HTTPException(422,'อ่านไฟล์รูปไม่ได้')


@app.get('/feedback-config')
def feedback_config():
    try:
        feedback_settings()
        ready = True
    except HTTPException:
        ready = False
    return {'configured':ready,'endpoint':'/feedback','format':'multipart/form-data',
            'image_limit_bytes_after_png_conversion':_FEEDBACK_IMAGE_MAX,
            'ratings_min':1,'ratings_max':5,
            'note':'configured ตรวจเฉพาะ Secrets ไม่ได้ยืนยันสิทธิ์ Sheets/Drive; ทดสอบจริงตามคู่มือ'}


@app.post('/feedback')
def submit_feedback(metadata: str = Form(..., max_length=12000),
                    original_image: UploadFile | None = File(None),
                    result_image: UploadFile | None = File(None)):
    acquired = _FEEDBACK_SLOTS.acquire(blocking=False)
    try:
        if not acquired:
            raise HTTPException(429,'กำลังบันทึกความคิดเห็นอื่น กรุณาลองใหม่',headers={'Retry-After':'5'})
        try:
            data = validate_feedback(json.loads(metadata))
        except (ValueError, TypeError) as exc:
            raise HTTPException(422,str(exc))
        url, secret = feedback_settings()
        images = {}
        if data['image_storage_consent']:
            images = {'original':feedback_image(original_image),'result':feedback_image(result_image)}
        # When false: do not read, forward, or store attached images.
        canonical = json.dumps({'metadata':data,'images':images},sort_keys=True,separators=(',',':'),ensure_ascii=False)
        payload = {'secret':secret,'metadata':data,'images':images,
                   'payload_hash':hashlib.sha256(canonical.encode('utf-8')).hexdigest()}
        try:
            upstream = requests.post(url,json=payload,timeout=(10,75))
            upstream.raise_for_status()
            answer = upstream.json()
        except (requests.RequestException,ValueError):
            raise HTTPException(502,'ยังยืนยันการบันทึกไม่ได้ ตรวจ Apps Script แล้วลองใหม่ด้วย experiment_id และคำตอบเดิม')
        if not isinstance(answer,dict) or answer.get('ok') is not True:
            code = answer.get('code') if isinstance(answer,dict) else None
            if code == 'CONFLICT':
                raise HTTPException(409,'ภาพนี้มีคำตอบที่บันทึกแล้วต่างจากคำตอบครั้งนี้ ไม่สร้างแถวซ้ำ')
            if code == 'BUSY':
                raise HTTPException(503,'ระบบเก็บข้อมูลกำลังทำงาน กรุณาลองใหม่ด้วยข้อมูลเดิม')
            raise HTTPException(502,'บันทึกไม่สำเร็จ ตรวจ Script Properties / สิทธิ์ Sheets และ Drive')
        if answer.get('experiment_id') != data['experiment_id'] or answer.get('result_key') != data['result_key']:
            raise HTTPException(502,'ผลยืนยันไม่ตรงกับการทดลอง')
        return {'saved':True,'experiment_id':data['experiment_id'],'result_key':data['result_key'],
                'duplicate':answer.get('duplicate') is True,
                'message':'ขอบคุณสำหรับความคิดเห็น'}
    finally:
        if acquired: _FEEDBACK_SLOTS.release()
        for upload in (original_image,result_image):
            if upload is not None: upload.file.close()

print('เพิ่ม /feedback และ /feedback-config แล้ว — รันเซลล์ server/ngrok ใหม่หลังรันเซลล์นี้')


In [ ]:
START_API = True
if START_API:
    import uvicorn, urllib.request
    from pyngrok import ngrok
    from google.colab import userdata
    token = userdata.get('NGROK_AUTH_TOKEN')
    if not token:
        raise ValueError('เพิ่ม NGROK_AUTH_TOKEN ใน Colab Secrets ก่อน')
    ngrok.set_auth_token(token)
    del token
    # หยุด server ตัวเดิมก่อน rerun ไม่สร้าง thread ชน port ซ้ำ
    old_server = globals().get('_api_server')
    old_thread = globals().get('_api_thread')
    if old_server is not None and old_thread is not None and old_thread.is_alive():
        old_server.should_exit = True
        old_thread.join(timeout=10)
        if old_thread.is_alive():
            raise RuntimeError('server เก่ายังมีงานค้าง รอให้เสร็จก่อนเปิดใหม่')
    _api_server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info'))
    _api_thread = threading.Thread(target=_api_server.run, daemon=True)
    _api_thread.start()
    for _ in range(40):
        if _api_server.started:
            break
        if not _api_thread.is_alive():
            raise RuntimeError('server เริ่มไม่ได้ ตรวจ port/logs')
        time.sleep(.25)
    if not _api_server.started:
        raise RuntimeError('server ยังไม่พร้อม')
    with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
        print('Local health:', response.status)
    previous = globals().get('_ngrok_tunnel')
    if previous is not None:
        ngrok.disconnect(previous.public_url)
    _ngrok_tunnel = ngrok.connect(8000)
    print('API URL:', _ngrok_tunnel.public_url)
    print('ดู schema:', _ngrok_tunnel.public_url+'/docs')


## 11. เปิด API + ngrok (เลือกทำ)
ตั้ง `START_API=True` และเพิ่ม `NGROK_AUTH_TOKEN` ใน Colab Secrets

## ข้อจำกัด
- การเพิ่มความคมช่วยเน้นรายละเอียดที่มีอยู่ ไม่รับประกันกู้เส้นผมที่หายไป
- segmentation อาจผิดพลาด; ลดความแรงเมื่อเห็นผิวหยาบ/ขอบเรือง
- ไม่มีการทดสอบโมเดลบน GPU ในขั้นแก้ไฟล์นี้ เวลาจริงดู native_sharp และ total_processing ในผลรัน


In [ ]:
RUN_FEEDBACK_SMOKE_TEST = False  # True writes one REAL test row, without images.
if RUN_FEEDBACK_SMOKE_TEST:
    import requests, uuid
    test_metadata = {
        'experiment_id': uuid.uuid4().hex, 'result_key': 'test_image',
        'experiment_type': 'both', 'hairstyle_id': 'TEST', 'hairstyle_name': '[TEST] ทรงผม',
        'hair_color_id': 'TEST', 'hair_color_name': '[TEST] สีผม',
        'mode': 'manual_smoke_test', 'seed': None, 'source_endpoint': '/feedback',
        'user_wait_seconds': 12.5, 'hairstyle_rating': 4, 'hair_color_rating': 5,
        'face_similarity_rating': 4, 'waiting_satisfaction_rating': 3,
        'overall_satisfaction_rating': 4, 'age_group': '18–24 ปี',
        'website_satisfaction_rating': 4,
        'suggestion': '[TEST] ลบแถวทดสอบได้', 'image_storage_consent': False,
    }
    for attempt in range(2):
        response = requests.post('http://127.0.0.1:8000/feedback',
            data={'metadata': json.dumps(test_metadata,ensure_ascii=False)}, timeout=110)
        response.raise_for_status()
        answer = response.json()
        assert answer.get('saved') is True, answer
        if attempt == 1: assert answer.get('duplicate') is True, answer
        print('ครั้งที่', attempt+1, answer)
    print('ตรวจ Sheets: ต้องเพิ่มเพียงหนึ่งแถว และลิงก์รูปว่าง')


## 12. ทดสอบก่อน–หลัง diffusion (เฉพาะ Colab)
ตั้ง `RUN_BEFORE_AFTER_TEST=True` ในเซลล์ถัดไป แล้วรันหลังส่วน 1–7 พร้อมแล้ว ไม่ต้องเปิด server หรือเรียก API
ถ้ามี B24 เดิม ใส่ `COMPARE_B24_PATH` เพื่อข้าม HairFastGAN; หากไม่มีให้อัปโหลดหน้า/ทรง/สีอย่างละหนึ่งภาพ
แสดงก่อนเข้า / diffusion ดิบ / ผลสุดท้าย และ crop ผมตำแหน่งเดียวกัน ไม่มีการแก้การทำงานเว็บ


In [ ]:
# ทดสอบเฉพาะใน Colab ไม่แก้ API ไม่ส่งภาพไปเว็บ
RUN_BEFORE_AFTER_TEST = True  # เปลี่ยนเป็น True แล้วรันเซลล์นี้
COMPARE_B24_PATH = ''  # มี hairfast_B24.png อยู่แล้ว: ใส่ path เพื่อข้ามการสร้าง B24
COMPARE_INPUT_PATHS = []  # หรือใส่ [face_path, shape_path, color_path]; ว่าง=อัปโหลด 3 รูป
COMPARE_SEED = 3407
COMPARE_STRENGTH = 0.15
COMPARE_STEPS = 30
COMPARE_GUIDANCE = 2.5
COMPARE_FACE_BLEND = 0.35
COMPARE_CROP = None  # กำหนด (left, top, right, bottom) หรือ None เพื่อเลือกกลางผมอัตโนมัติ


def test_before_after_diffusion():
    import io, tempfile, time
    from pathlib import Path
    import numpy as np
    from PIL import Image, ImageOps
    import matplotlib.pyplot as plt
    from IPython.display import display
    from scipy.ndimage import distance_transform_edt
    required=('hair_fast','inpaint_pipe','GPU_LOCK','tensor_to_pil','detail_only_masks',
              'refine_b24_details','transfer_luminance_detail','resolve_inpaint_settings','run_detail_core','finish_raw_background')
    missing=[name for name in required if name not in globals()]
    if missing:raise RuntimeError('รันส่วน 1–7 ให้พร้อมก่อน ขาด: '+', '.join(missing))
    settings=resolve_inpaint_settings(COMPARE_STRENGTH,COMPARE_STEPS,COMPARE_GUIDANCE)
    resolve_joint_settings(COMPARE_FACE_BLEND)
    if not 0<=int(COMPARE_SEED)<=4294967295:raise ValueError('seed ไม่ถูกต้อง')
    timings={}
    def clock():
        if torch.cuda.is_available():torch.cuda.synchronize()
        return time.perf_counter()
    with tempfile.TemporaryDirectory(prefix='hfg_compare_') as temp:
        inputs=[]
        if not COMPARE_B24_PATH:
            if COMPARE_INPUT_PATHS:
                if len(COMPARE_INPUT_PATHS)!=3:raise ValueError('ต้องมี path หน้า/ทรง/สี ตามลำดับ')
                inputs=[Path(v) for v in COMPARE_INPUT_PATHS]
                if not all(v.is_file() for v in inputs):raise FileNotFoundError('ตรวจ path รูปทั้งสามใบ')
            else:
                from google.colab import files
                for role,label in [('face','รูปหน้า'),('shape','รูปทรงผม'),('color','รูปสีผม')]:
                    print('อัปโหลด '+label+' 1 ภาพ')
                    upload=files.upload()
                    if len(upload)!=1:raise ValueError('เลือกครั้งละหนึ่งภาพ')
                    path=Path(temp)/(role+'.png')
                    with Image.open(io.BytesIO(next(iter(upload.values())))) as im:
                        ImageOps.exif_transpose(im).convert('RGB').save(path)
                    inputs.append(path)
        cached_before=None
        if COMPARE_B24_PATH:
            with Image.open(COMPARE_B24_PATH) as im:cached_before=im.convert('RGB').copy()
        core_result=run_detail_core(inputs=inputs,before=cached_before,seed=COMPARE_SEED,
            strength=COMPARE_STRENGTH,steps=COMPARE_STEPS,guidance=COMPARE_GUIDANCE,
            face_blend=COMPARE_FACE_BLEND)
        before=core_result['before'];raw=core_result['raw_diffusion'];after=core_result['after']
        masks=core_result['masks'];timings=core_result['timings_seconds']
    # Compare identical coordinates and display sizes; don't sharpen previews.
    before_sharp=core_result['before_sharp']
    images=[before,raw,before_sharp,after]
    titles=['1. B24 before diffusion','2. Raw diffusion (diagnostic only)',
            '3. Original final (no sharp)', '4. Native sharp final']
    fig,axes=plt.subplots(1,4,figsize=(20,5))
    for ax,im,title in zip(axes,images,titles):
        ax.imshow(im,interpolation='nearest');ax.set_title(title);ax.axis('off')
    fig.tight_layout();display(fig);plt.close(fig)
    if COMPARE_CROP is None:
        d=distance_transform_edt(np.pad(masks['hair'],1))[1:-1,1:-1]
        cy,cx=np.unravel_index(np.argmax(d),d.shape)
        side=min(256,before.width,before.height)
        left=max(0,min(before.width-side,int(cx)-side//2))
        top=max(0,min(before.height-side,int(cy)-side//2))
        box=(left,top,left+side,top+side)
    else:
        if len(COMPARE_CROP)!=4:raise ValueError('crop ต้องมี 4 ค่า')
        box=tuple(int(v) for v in COMPARE_CROP)
        if not (0<=box[0]<box[2]<=before.width and 0<=box[1]<box[3]<=before.height):
            raise ValueError('crop อยู่นอกภาพ')
    fig,axes=plt.subplots(1,4,figsize=(20,5))
    for ax,im,title in zip(axes,images,titles):
        ax.imshow(im.crop(box),interpolation='nearest');ax.set_title(title+' / hair crop');ax.axis('off')
    fig.tight_layout();display(fig);plt.close(fig)
    print('Crop coordinates:',box)
    print('เวลารัน (ไม่รวมอัปโหลดและแสดงภาพ):',{k:round(v,2) for k,v in timings.items()})
    print('ภาพ 3 = ผลเดิมก่อนเพิ่มความคม; ภาพ 4 = ผลใหม่ส่งเว็บ ขนาดเดิม ไม่มี SR')
    print('Native sharp:',core_result['native_sharp'])
    # Keep originals in memory for inspection; no Drive writes, no API mutations.
    return dict(before=before,raw_diffusion=raw,before_sharp=before_sharp,after=after,crop=box,
                sharp_labels=core_result['sharp_labels'],native_sharp=core_result['native_sharp'],
                settings=settings,seed=COMPARE_SEED,timings_seconds=timings)

if RUN_BEFORE_AFTER_TEST:
    before_after_test=test_before_after_diffusion()
else:
    print('ตั้ง RUN_BEFORE_AFTER_TEST=True แล้วรันเซลล์นี้เพื่อเปรียบเทียบภาพ')


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import files
import tempfile

if "before_after_test" not in globals():
    raise RuntimeError("รันเซลล์เปรียบเทียบภาพให้เสร็จก่อน")

image_items = [
    ("before", "01_B24_ก่อนเข้า_Diffusion"),
    ("raw_diffusion", "02_Diffusion_ก่อนปรับพื้นหลัง"),
    ("before_sharp", "03_ผลเดิม_ก่อนเพิ่มความคม"),
    ("after", "04_ผลใหม่_เพิ่มความคมผมและหน้า_1024"),
]

for key, _ in image_items:
    if key not in before_after_test:
        raise RuntimeError(f"ไม่พบภาพ: {key}")

output_dir = Path(tempfile.mkdtemp(prefix="comparison_", dir="/content"))
timestamp = datetime.now(ZoneInfo("Asia/Bangkok")).strftime("%Y%m%d_%H%M%S")

for key, name in image_items:
    image_path = output_dir / f"{name}_{timestamp}.png"

    # บันทึกภาพเต็มความละเอียดเดิม ไม่ย่อ ไม่เพิ่มความคมซ้ำ
    before_after_test[key].save(
        image_path, format="PNG", compress_level=1
    )

    print("กำลังดาวน์โหลด:", image_path.name)
    files.download(str(image_path))

## 13. ลองความแรงใหม่จากผลเดิม ไม่รัน HairFastGAN / Diffusion ซ้ำ
ปรับค่าด้านล่างแล้วรัน ดูภาพเดิมเทียบภาพใหม่ ผลนี้เก็บใน before_after_test และดาวน์โหลดได้ด้วยเซลล์ดาวน์โหลดด้านบน


In [ ]:
# ใช้เฉพาะเมื่อส่วน 12 ทำงานเสร็จแล้ว
RUN_RETUNE_SHARP = False
if RUN_RETUNE_SHARP:
    import time
    import matplotlib.pyplot as plt
    from IPython.display import display
    NATIVE_SHARP_ENABLED = True
    NATIVE_HAIR_AMOUNT = 0.65  # ลอง 0.45 / 0.65 / 0.85
    NATIVE_FACE_AMOUNT = 0.25  # ลอง 0.15 / 0.25 / 0.35
    if 'before_after_test' not in globals() or 'sharp_labels' not in before_after_test:
        raise RuntimeError('รันส่วน 12 ของไฟล์นี้ก่อน')
    base=before_after_test['before_sharp']  # always original, never sharpen cumulatively
    start=time.perf_counter()
    tuned,info=finish_native_sharp(base,before_after_test['sharp_labels'])
    elapsed=time.perf_counter()-start
    before_after_test['after']=tuned
    before_after_test['native_sharp']=info
    before_after_test['timings_seconds']['native_sharp']=elapsed
    before_after_test['timings_seconds']['total_processing']=sum(
        before_after_test['timings_seconds'].get(k,0.)
        for k in ('B24','mask','diffusion','raw_background_finish','native_sharp'))
    fig,axes=plt.subplots(1,2,figsize=(12,6))
    for ax,im,title in zip(axes,[base,tuned],['Original final','Native sharp']):
        ax.imshow(im,interpolation='nearest');ax.set_title(title);ax.axis('off')
    display(fig);plt.close(fig)
    print('เพิ่มความคมอย่างเดียว:',round(elapsed,3),'วินาที',info)
    print('ดาวน์โหลดผลใหม่ได้โดยรันเซลล์ดาวน์โหลด 4 รูปอีกครั้ง')
